In [1]:
from collections import Counter
from huffman import *
import networkx as nx
import matplotlib.pyplot as plt
import heapq

# Introduction

Data compression plays a crucial role in modern computing, reducing storage requirements and improving transmission efficiency.

In this tutorial, we explore the Huffman Compression Algorithm — a widely used lossless compression method. The algorithm is based on assigning variable-length binary codes to characters based on their frequencies.

## Objectives
- Understand the mathematical intuition behind Huffman coding
- Implement the algorithm from scratch in Python
- Analyze its efficiency
- Compare it with fixed-length encoding

# Problem Formulation

Given a string of characters, we want to encode it into binary form using the least number of bits possible.

Constraints:
- Each character must have a unique code
- Codes must be prefix-free (no code is a prefix of another)

Goal:
Minimize the total number of bits required to encode the message.

# Mathematical Background

Huffman coding is a greedy algorithm.

Key idea:
- Characters with higher frequency should have shorter codes
- Characters with lower frequency should have longer codes

Let:
- f(c) = frequency of character c
- l(c) = length of code for character c

We minimize:

    Total Cost = Σ f(c) * l(c)

This is related to entropy in information theory.

# Huffman Algorithm

Steps:
1. Count frequency of each character
2. Create a priority queue (min-heap)
3. Insert all characters as nodes
4. While there is more than one node:
    - Extract two nodes with smallest frequency
    - Merge them into a new node
    - Insert back into the heap
5. The remaining node is the root of the Huffman tree
6. Traverse the tree to assign codes

In [2]:
class Node:
    def __init__(self, char, freq):
        self.char = char
        self.freq = freq
        self.left = None
        self.right = None

    def __lt__(self, other):
        return self.freq < other.freq

In [3]:
def build_frequency_table(text):
    return Counter(text)

## Huffman Tree Visualization

To better understand the structure of the Huffman tree, we visualize it as a graph.

- Each node shows frequency (and character if leaf)
- Left edges = 0
- Right edges = 1

In [4]:
def build_graph(node, graph=None, parent=None, edge_label=""):
    if graph is None:
        graph = nx.DiGraph()

    if node is None:
        return graph

    node_label = f"{node.char}:{node.freq}" if node.char else f"{node.freq}"
    node_id = str(id(node))

    graph.add_node(node_id, label=node_label)

    if parent:
        graph.add_edge(parent, node_id, label=edge_label)

    if node.left:
        build_graph(node.left, graph, node_id, "0")
    if node.right:
        build_graph(node.right, graph, node_id, "1")

    return graph


def draw_tree(root):
    graph = build_graph(root)

    print("Nodes:", graph.nodes())
    print("Edges:", graph.edges())

    pos = nx.spring_layout(graph, seed=42)
    edge_labels = nx.get_edge_attributes(graph, 'label')
    labels = nx.get_node_attributes(graph, 'label')

    plt.figure(figsize=(12, 8))
    nx.draw(graph, pos, labels=labels, with_labels=True,
            node_size=2000, node_color="lightblue", font_size=10)
    nx.draw_networkx_edge_labels(graph, pos, edge_labels=edge_labels)

    plt.show()

In [5]:
def build_huffman_tree(freq_table):
    heap = [Node(char, freq) for char, freq in freq_table.items()]
    heapq.heapify(heap)

    while len(heap) > 1:
        left = heapq.heappop(heap)
        right = heapq.heappop(heap)

        merged = Node(None, left.freq + right.freq)
        merged.left = left
        merged.right = right

        heapq.heappush(heap, merged)

    return heap[0]

In [6]:
def generate_codes(node, prefix="", codebook={}):
    if node is None:
        return

    if node.char is not None:
        codebook[node.char] = prefix

    generate_codes(node.left, prefix + "0", codebook)
    generate_codes(node.right, prefix + "1", codebook)

    return codebook

In [7]:
def encode(text, codebook):
    return ''.join(codebook[char] for char in text)

In [8]:
def decode(encoded_text, root):
    decoded = ""
    current = root

    for bit in encoded_text:
        current = current.left if bit == '0' else current.right

        if current.char is not None:
            decoded += current.char
            current = root

    return decoded

In [9]:
text = "this is an example for huffman encoding"

freq_table = build_frequency_table(text)
root = build_huffman_tree(freq_table)
codebook = generate_codes(root)

encoded = encode(text, codebook)
decoded = decode(encoded, root)

print("Original:", text)
print("Encoded:", encoded)
print("Decoded:", decoded)

Original: this is an example for huffman encoding
Encoded: 0101001001001001010110010010101111100010111100111011110011100000110111101011101110010110010101001100011011101001111110001011110000011111100101011100100010001
Decoded: this is an example for huffman encoding


# Analysis

Original size:
Each character = 8 bits

Compressed size:
Length of encoded string

Compression ratio:

    ratio = compressed_bits / original_bits

Huffman coding achieves better compression when:
- There is high variation in character frequencies

In [10]:
original_bits = len(text) * 8
compressed_bits = len(encoded)

print("Original bits:", original_bits)
print("Compressed bits:", compressed_bits)
print("Compression ratio:", compressed_bits / original_bits)

Original bits: 312
Compressed bits: 157
Compression ratio: 0.5032051282051282


# Comparison

Fixed-length encoding:
- All characters use same number of bits

Huffman encoding:
- Variable-length
- More efficient

Trade-offs:
+ Better compression
- Requires tree structure for decoding

# Extensions

Possible improvements:
- Compress real files
- Visualize the Huffman tree
- Compare with other algorithms (e.g., Run-Length Encoding)
- Measure runtime complexityv

# Conclusion

We successfully implemented the Huffman Compression Algorithm and demonstrated how it reduces the size of data using variable-length encoding.

Key takeaways:
- Greedy algorithms can produce optimal results
- Data structure choice (heap) is critical
- Compression depends heavily on data distribution

# Huffman Compression Algorithm redmi

This project demonstrates the implementation of Huffman Coding in Python.

## Features
- Full algorithm implementation
- Visualization
- Entropy analysis
- Compression metrics

## How to run
Open the Jupyter notebook and execute all cells.

In [11]:
class Node:
    def __init__(self, char, freq):
        self.char = char
        self.freq = freq
        self.left = None
        self.right = None

def print_tree(node, indent="", position="root"):
    if node is not None:
        label = f"{node.char}:{node.freq}" if node.char else f"{node.freq}"
        print(indent + position + " -> " + label)

        print_tree(node.left, indent + "   ", "L")
        print_tree(node.right, indent + "   ", "R")


# пример
root = Node(None, 5)
root.left = Node('a', 2)
root.right = Node('b', 3)

print_tree(root)

root -> 5
   L -> a:2
   R -> b:3
